# 28B Heartbeat: within-filing signals, detect, decide, deduplicate
Every signal compares this period's schedule with the prior fiscal year-end schedule **in the same filing**. Pairing the two is the hard part, and it is yours to design.

In [ ]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

In [ ]:
heads = json.load(open(FX / "release_heads.json"))
tick = heartbeat.release_tick(heads["last_seen"], lambda url: heads["heads"].get(url, 404), P["sec"]["base_url"])
print(tick)

In [ ]:
pairs = signals.pair_exact(PANEL)
print("Exact pairing (same filing, identical identifier):")
print(signals.pairing_report(PANEL, pairs).to_string(index=False))
ex = PANEL[PANEL["name"].str.contains("GOLF") & PANEL["identifier"].str.contains("Cedar Ridge")]
print("\nWhy Golf pairs nothing: the identifier carries the all-in rate, which moved with SOFR")
print(ex[["period_role", "identifier"]].to_string(index=False))

In [ ]:
found = signals.find_signals(PANEL, M)
print(found[["name", "rule", "identifier", "row", "before", "after"]].to_string(index=False))
if P["data"]["use_fixture"]:
    truth = pd.read_csv(ROOT / P["data"]["truth"], dtype=str)
    sc = evals.signal_scores(found, truth)
    print(f"\nAgainst the planted events: precision {sc['precision']:.2f}, recall {sc['recall']:.2f} "
          f"({sc['tp']} found, {sc['fp']} false, {sc['fn']} missed)")
    print(pd.DataFrame(sc["by_rule"]).T)

In [ ]:
problems = panel.check_mandates(MANDATES, PANEL)    # a misspelled BDC name would route nothing, silently
assert not problems, problems
held = {m: list(v["positions"]) for m, v in MANDATES.items()}
routed = heartbeat.route(found, held)
seen = set()
early = [a for m, s in routed.items() for a in heartbeat.decide(s, m, seen, NOW)]
print(f"At {NOW:%H:%M} (quiet hours):", pd.Series([a["action"] for a in early]).value_counts().to_dict(), "| per mandate:",
      pd.Series([a["mandate"] for a in early]).value_counts().to_dict())
again = [a for m, s in routed.items() for a in heartbeat.decide(s, m, seen, NOW.replace(hour=9))]
print("Same signals at 09:00:", again, "(deduplicated)")
recent = json.load(open(FX / "edgar_recent.json"))
for bdc, cik in REG["bdc_cik"].items():
    new = heartbeat.new_filings(recent.get(cik, {"form": [], "items": [], "accessionNumber": [], "acceptanceDateTime": []}), seen=set())
    for m in MANDATES:
        for d in heartbeat.filing_decisions(new, bdc, held[m], m, seen, NOW.replace(hour=9)):
            print(f"  {m}: {d['form']:5s} {d['accn']} {bdc.replace('FIXTURE ', ''):24s} -> {d['action']}")

Exact pairing finds 6 of the 9 planted events with no false alarms: precision 1.00, recall 0.67. It misses Golf's two markdowns (the all-in rate in the identifier changed) and Juliett's extension (the maturity in the identifier changed). Two traps punish a careless fix: Golf's Granite Peak has a term loan and a delayed-draw tranche, only one of them marked down, and Hotel's Redwood Valley fell 4.5 points, under the 5-point threshold.

**Your turn.** Write your own pairing function, pass it as `signals.find_signals(panel, m, pair=your_pairing)`, and measure it: precision and recall on the fixture, plus a hand-checked sample of pairs from your real panel. Then build the four DAGs, deliver alerts and the brief, and measure latency, alerts per mandate-week and idle cost (Part 5).